# DeepSeekMath: dissecting the GRPO objective

**DeepSeekMath: Pushing the Limits of Mathematical Reasoning in Open Language Models** · Shao et al. · 2024

[Original paper](https://arxiv.org/abs/2402.03300)

**Learning goal:** Separate group advantages, old policy, reference KL, and response masks; test clipping and constant rewards.

## Before starting

Introductory GRPO, log-probabilities, masks, and gradients.

Allow roughly 40–60 minutes including exercises. CPU, PyTorch, and Matplotlib; no model or dataset downloads. This is a study-time estimate, not a runtime benchmark.

## Problem and idea

GRPO uses a response group per prompt to estimate relative quality without a separate value model. Do not mix prompts when normalizing rewards. Old policy collected the rollout; the reference supplies regularization when KL is enabled. These roles differ.

Even formal, verifiable rewards can be exploited. Falling policy loss does not establish improved reasoning. We use fixed synthetic responses and do not reproduce generation, mathematical verification, or the paper dataset.

## Reproduction boundary

Objective and diagnostic invariants only; not a full GRPO trainer, DeepSeekMath training, or mathematical benchmark reproduction.

The code is an original educational implementation of mathematical ideas from the paper, not a copy of an author repository. Passing assertions verifies the stated implementation properties, not the paper’s results.

## Formula → tensors

`A_i = (r_i - mean(group)) / max(std(group), eps)`. Constant within-group rewards give zero advantages. We explicitly use population std (`unbiased=False`); trainer conventions can differ.

`ratio = exp(logpi - logpi_old)`; surrogate=`min(ratio*A, clip(ratio,1-e,1+e)*A)`. The token KL estimator is `exp(logref-logpi) - (logref-logpi) - 1`. Average valid tokens per response, then responses. Detach old/reference log-probabilities.

In [ ]:
%matplotlib inline
import math
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(42)
plt.rcParams.update({'figure.figsize': (7, 3), 'axes.spines.top': False, 'axes.spines.right': False})
print('PyTorch:', torch.__version__, '| CPU | seed=42')

## Step 1

Two prompts with three responses each. The second group has constant rewards and no relative signal.

In [ ]:
rewards = torch.tensor([[0., 1., 3.], [2., 2., 2.]])
advantage = (rewards - rewards.mean(-1, keepdim=True)) / rewards.std(-1, keepdim=True, unbiased=False).clamp_min(1e-6)
assert torch.isfinite(advantage).all()
assert torch.equal(advantage[1], torch.zeros(3))
assert torch.allclose(advantage.mean(-1), torch.zeros(2), atol=1e-6)
print('advantages:', advantage)
mask = torch.tensor([[[1, 1, 0], [1, 1, 1], [1, 0, 0]], [[1, 1, 0], [1, 1, 1], [1, 0, 0]]], dtype=torch.float)

## Step 2

Compute a token-level objective. Free log-probability variables are a numerical unit experiment, not a fully normalized policy.

In [ ]:
old_logp = torch.full((2, 3, 3), -2.)
reference_logp = old_logp.clone()
current_logp = old_logp.clone().requires_grad_()
def objective(current):
    ratio = (current - old_logp).exp()
    A = advantage[..., None]
    surrogate = torch.minimum(ratio * A, ratio.clamp(0.8, 1.2) * A)
    difference = reference_logp - current
    kl = difference.exp() - difference - 1
    token_objective = surrogate - 0.04 * kl
    loss = -((token_objective * mask).sum(-1) / mask.sum(-1).clamp_min(1)).mean()
    return loss, kl
loss, kl = objective(current_logp); loss.backward()
assert torch.equal(kl, torch.zeros_like(kl))
assert torch.equal(current_logp.grad[mask == 0], torch.zeros_like(current_logp.grad[mask == 0]))
assert torch.equal(current_logp.grad[1], torch.zeros_like(current_logp.grad[1]))
print('initial loss:', loss.item(), 'gradient:', current_logp.grad)

## Step 3

Plot clipping for positive and negative advantages, independently of KL.

In [ ]:
ratios = torch.linspace(0.4, 1.6, 121)
for A, color in [(1., '#d73535'), (-1., '#777777')]:
    clipped = torch.minimum(ratios * A, ratios.clamp(0.8, 1.2) * A)
    plt.plot(ratios, clipped, label=f'advantage={A}', color=color)
plt.axvline(0.8, ls=':', color='gray'); plt.axvline(1.2, ls=':', color='gray')
plt.xlabel('policy ratio'); plt.ylabel('clipped surrogate'); plt.legend(); plt.show()
varied = old_logp.clone(); varied[0, 0, 2] += 5
assert torch.allclose(objective(varied)[0], objective(old_logp)[0])

## Reading the result

For A>0 the plateau is above 1+epsilon; for A<0 it is below 1-epsilon. Clipping neither prevents every large parameter change nor guarantees bounded KL. Zero advantage does not remove KL pressure if policy differs from reference.

## Independent experiments

1. Normalize across the full batch and identify the semantic error.
2. Change the second group’s current log-probs and inspect KL gradients.
3. Compare token averages with response averages for different lengths.

Write a prediction first, then change one factor at a time. Reset the kernel and rerun all cells for a controlled comparison. Save a copy for your changes.

## Reasoning check — after experimenting

Batch normalization compares responses to different tasks. Reference divergence can supply a KL signal even with zero reward advantage. Token averages weight longer answers more; response averages give each response equal weight.

## What to record in your lab notes

- Hypothesis and changed parameter.
- Seed, data size, step budget, and precision.
- Before/after diagnostics, plot, and explanation of unexpected results.
- What this experiment **does not** establish.

Use “Open Studio workspace” above the lesson to continue in the related tools.